## Spatial Join

In [2]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [3]:
base = Path(r"C:/Your_path")

boroughs = gpd.read_file(base / "geodata/GM_lad.gpkg")
stops = gpd.read_file(base / "geodata/Metrolink_Stops_Functional.json")

print(boroughs.crs)
print(stops.crs)

EPSG:4326
EPSG:4326


Set Coordinate Reference System to OSGB36 / British National Grid

In [4]:
boroughs = boroughs.to_crs(epsg=27700)
stops = stops.to_crs(epsg=27700)


Spatial join: assign each stop to the borough it falls within

In [5]:
stops_in_borough = gpd.sjoin(
    stops,
    boroughs,
    how="inner",
    predicate="within"
)

stops_in_borough.head()

,description,type,name,validFrom,currentStatus,comments,stationCode,ticketZone,stroke,geometry,index_right,LAD22CD,LAD22NM,BNG_E,BNG_N,LONG,LAT,GlobalID
0,Heaton Park Metrolink stop.,railwayStop,Heaton Park,19920406,functional,NaN,HPK,3,#000000,POINT (382382.361 403792.292),8,E08000002,Bury,379658,410768,-2.30880,53.59310,528b15a7-dba9-46fb-90c3-61505a610756
1,Bowker Vale Metrolink stop.,railwayStop,Bowker Vale,19920406,functional,NaN,BKV,2/3,#000000,POINT (383530.343 403163.727),9,E08000003,Manchester,384591,397063,-2.23359,53.47009,dcb3ecf1-0228-43d0-b65f-f40a4e801466
2,Crumpsall Metrolink stop.,railwayStop,Crumpsall,19920406,functional,NaN,CRP,2,#000000,POINT (384114.233 402316.208),9,E08000003,Manchester,384591,397063,-2.23359,53.47009,dcb3ecf1-0228-43d0-b65f-f40a4e801466
3,Bury Metrolink stop.,railwayStop,Bury,19920406,functional,NaN,BRY,4,#000000,POINT (380431.881 410525.928),8,E08000002,Bury,379658,410768,-2.30880,53.59310,528b15a7-dba9-46fb-90c3-61505a610756
4,Radcliffe Metrolink stop.,railwayStop,Radcliffe,19920406,functional,NaN,RAD,4,#000000,POINT (378833.247 407354.405),8,E08000002,Bury,379658,410768,-2.30880,53.59310,528b15a7-dba9-46fb-90c3-61505a610756


Count tram stops per borough

In [7]:
borough_counts = (
    stops_in_borough
    .groupby("LAD22NM")
    .size()
    .reset_index(name="n_stops")
    .sort_values("n_stops", ascending=False)
)

Boroughs not served by the tram network

In [8]:
not_served = (
    boroughs[["LAD22NM"]]
    .merge(
        stops_in_borough[["LAD22NM"]].drop_duplicates(),
        on="LAD22NM",
        how="left",
        indicator=True
    )
    .query('_merge == "left_only"')
    [["LAD22NM"]]
)

Inspect outputs

In [9]:
print("\nStops joined to boroughs:")
print(stops_in_borough[["name", "stationCode", "LAD22NM"]].head())

print("\nStops per borough:")
print(borough_counts)

print("\nBoroughs not served:")
print(not_served)


Stops joined to boroughs:
          name stationCode     LAD22NM
0  Heaton Park         HPK        Bury
1  Bowker Vale         BKV  Manchester
2    Crumpsall         CRP  Manchester
3         Bury         BRY        Bury
4    Radcliffe         RAD        Bury

Stops per borough:
      LAD22NM  n_stops
1  Manchester       42
6    Trafford       18
2      Oldham       10
4     Salford       10
5    Tameside        7
0        Bury        6
3    Rochdale        6

Boroughs not served:
                  LAD22NM
0              Warrington
1   Blackburn with Darwen
2           Cheshire East
3               High Peak
4                 Chorley
5              Rossendale
6         West Lancashire
7                  Bolton
13              Stockport
16                  Wigan
17             St. Helens
18             Calderdale
19               Kirklees
